In [2]:
import os
import numpy as np
import librosa
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# ===== SETTINGS =====
DATASET_PATH = r"D:\FYP\dataset\audio"
SAMPLE_RATE = 22050
MAX_PAD_LEN = 174
N_MFCC = 40

# ===== FEATURE EXTRACTION =====
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC
        )

        # padding / trimming
        if mfcc.shape[1] < MAX_PAD_LEN:
            pad_width = MAX_PAD_LEN - mfcc.shape[1]
            mfcc = np.pad(mfcc,
                          pad_width=((0, 0), (0, pad_width)),
                          mode='constant')
        else:
            mfcc = mfcc[:, :MAX_PAD_LEN]

        return mfcc

    except Exception as e:
        print("Error:", file_path)
        return None


# ===== LOAD DATA =====
X = []
y = []

classes = ["real", "fake"]

for label, cls in enumerate(classes):
    folder = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(folder):
        if file.endswith(".wav"):
            path = os.path.join(folder, file)

            features = extract_features(path)

            if features is not None:
                X.append(features)
                y.append(label)

# convert to numpy
X = np.array(X)
y = to_categorical(y, num_classes=2)

# ===== RESHAPE (IMPORTANT) =====
# (samples, time_steps, features)
X = X.reshape(X.shape[0], MAX_PAD_LEN, N_MFCC)

# ===== SPLIT =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ===== MODEL (LSTM) =====
model = Sequential()

model.add(LSTM(64, return_sequences=True,
               input_shape=(MAX_PAD_LEN, N_MFCC)))
model.add(Dropout(0.3))

model.add(LSTM(64))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dense(2, activation='softmax'))

# ===== COMPILE =====
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ===== CALLBACK =====
early_stop = EarlyStopping(
    patience=5,
    restore_best_weights=True
)

# ===== TRAIN =====
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

# ===== EVALUATE =====
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

# ===== SAVE =====
model.save("deepfake_audio_model.h5")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 174, 64)           26880     
                                                                 
 dropout (Dropout)           (None, 174, 64)           0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                33024     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 2)                 130       
                                                                 
Total params: 64194 (250.76 KB)
Trainable params: 6419

C:\Users\Zee\AppData\Roaming\Python\Python310\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
